# End-to-End Apache Iceberg SQL Tutorial

Managing a high-throughput **E-commerce Order Lifecycle Platform** requires a data architecture that can handle continuous incremental updates, regulatory auditing, and strict data quality checks without halting production reads. Apache Iceberg solves this by decoupling physical storage from logical table structures using a robust metadata tree.

This notebook walks through a complete implementation of an `orders` table utilizing Iceberg's advanced catalog procedures, Merge-on-Read mechanics, and the Write-Audit-Publish (WAP) pattern. It is configured to run out-of-the-box on your `demo` Iceberg catalog.

In [1]:
from pyspark.sql import SparkSession

# Initialize Spark Session (Inherits configurations automatically from spark-defaults.conf)
spark = SparkSession.builder.appName('Iceberg-SQL-Tutorial').getOrCreate()
# Silence verbose Spark/Iceberg warnings
spark.sparkContext.setLogLevel("ERROR")
print('Spark Session initialized successfully.')

Spark Session initialized successfully.


26/11/24 00:39:52 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


# Create a dedicated namespace


In [2]:
# Create a dedicated namespace
spark.sql("CREATE NAMESPACE IF NOT EXISTS demo.ecommerce")

DataFrame[]

## 1. Scenario Setup & DDL

First, we isolate the domain by creating a dedicated namespace. 

When creating the table, we leverage **Hidden Partitioning**. Instead of forcing engineers to extract a physical `order_date` column, Iceberg's `days()` transform dynamically derives the partition from `order_timestamp`. 

We explicitly configure **Merge-on-Read (MoR)** for row-level operations. MoR writes updates and deletes as separate differential files rather than rewriting entire Parquet data files on every mutation, dramatically improving write latency.

- 'format-version' = '2',                        -- Enables row-level updates, deletes, and MoR (Delete files (positional and equality):                                                    Tracks deleted rows separately from existing data files so writers can record row-level changes quickly.)
- 'write.update.mode' = 'merge-on-read',         -- Writes updates as delete + new data files
- 'write.delete.mode' = 'merge-on-read',         -- Writes deletes as positional delete files
- 'write.merge.mode' = 'merge-on-read',          -- Optimises MERGE INTO write latency
- 'write.parquet.compression-codec' = 'zstd'     -- Higher compression ratio than default snappy

###  Positional Deletes

The query engine scans the affected data file(s) to find matching rows and identifies their exact zero-based row offsets (`0, 1, 2, ...`).
        
    - It writes a compact Parquet delete file containing two core columns: `file_path` (the exact path of the target data file) and `pos` (the row index inside that file).
        
    - The original large data file remains untouched on disk.


In [10]:
# Create a dedicated namespace
spark.sql("CREATE NAMESPACE IF NOT EXISTS demo.ecommerce")

# Drop table if it already exists for clean re-runs
spark.sql("DROP TABLE IF EXISTS demo.ecommerce.orders")

# Create the Iceberg table
spark.sql("""
CREATE TABLE demo.ecommerce.orders (
    order_id BIGINT,
    customer_id BIGINT,
    order_timestamp TIMESTAMP,
    status STRING,
    total_amount DECIMAL(10, 2)
)
USING iceberg
PARTITIONED BY (days(order_timestamp))
TBLPROPERTIES (
    'format-version' = '2',                        
    'write.update.mode' = 'merge-on-read',         
    'write.delete.mode' = 'merge-on-read',         
    'write.merge.mode' = 'merge-on-read',          
    'write.parquet.compression-codec' = 'zstd'     
)
""")
print("Namespace and Table created successfully.")

Namespace and Table created successfully.


## CHECK for the Matadata Changes

- **Catalog Layer**: Maintains the central, atomic pointer to the current active `.metadata.json` file.
- **Metadata File (`.metadata.json`)**: Contains overall table parameters, schemas, partition specs, properties, snapshot arrays, and the `current-snapshot-id` pointer.
- **Manifest List (`snap-*.avro`)**: Represents a single committed snapshot, pointing to a list of manifest files with partition ranges and summary metrics.
- **Manifest File (`m-*.avro`)**: Contains the explicit list of data/delete file paths, status flags (`ADDED`, `EXISTING`, `DELETED`), and column-level min/max statistics for query pruning.
- **Data Files (`.parquet`, `.orc`, `.avro`)**: Immutable storage files holding raw record rows.

## 

# 1.2 : Data Ingestion via files

Ingest the Data from the file using the code file

## --------------------------------------------------------------------------
## Check for the partitions in the table after the data inserted into the Tables

In [17]:
spark.sql('''
SELECT
  partition,
  spec_id,
  record_count,
  file_count,
  total_data_file_size_in_bytes
FROM demo.ecommerce.orders.partitions
ORDER BY partition''').show()

+------------+-------+------------+----------+-----------------------------+
|   partition|spec_id|record_count|file_count|total_data_file_size_in_bytes|
+------------+-------+------------+----------+-----------------------------+
|{2026-07-15}|      0|         318|         1|                         5242|
|{2026-08-20}|      0|         324|         1|                         5315|
|{2026-09-10}|      0|         358|         1|                         5619|
+------------+-------+------------+----------+-----------------------------+



In [20]:
%%sql
SELECT * FROM demo.ecommerce.orders.history;       

made_current_at,snapshot_id,parent_id,is_current_ancestor
2026-11-24 01:45:28.342000,575426542183088885,None,True


In [12]:
print('--- Iceberg Orders Physical Files & Hidden Partitions ---')
spark.sql('''
    SELECT file_path, spec_id, partition, record_count, ROUND(file_size_in_bytes / 1024, 2) AS size_kb
    FROM demo.ecommerce.orders.files
''').show(truncate=False)

print('--- Sample Join: Revenue by Order Date & Customer Segment ---')
spark.sql('''
    SELECT
        DATE(o.order_timestamp) AS order_date,
        c.segment,
        COUNT(o.order_id) AS total_orders,
        ROUND(SUM(o.total_amount), 2) AS total_revenue
    FROM demo.ecommerce.orders o
    JOIN demo.ecommerce.customers c
        ON o.customer_id = c.customer_id
    GROUP BY DATE(o.order_timestamp), c.segment
    ORDER BY order_date, c.segment
''').show()

--- Iceberg Orders Physical Files & Hidden Partitions ---
+----------------------------------------------------------------------------------------------------------------------------------+-------+------------+------------+-------+
|file_path                                                                                                                         |spec_id|partition   |record_count|size_kb|
+----------------------------------------------------------------------------------------------------------------------------------+-------+------------+------------+-------+
|s3a://warehouse/ecommerce/orders/data/order_timestamp_day=2026-07-15/00000-11-7ed76c1c-42ea-4084-bd4a-0f5c9a538d0c-0-00001.parquet|0      |{2026-07-15}|318         |5.12   |
|s3a://warehouse/ecommerce/orders/data/order_timestamp_day=2026-08-20/00000-11-7ed76c1c-42ea-4084-bd4a-0f5c9a538d0c-0-00003.parquet|0      |{2026-08-20}|324         |5.19   |
|s3a://warehouse/ecommerce/orders/data/order_timestamp_day=2026-09-

In [24]:
%%sql

SELECT
        DATE(o.order_timestamp) AS order_date,
        c.segment,
        COUNT(o.order_id) AS total_orders,
        ROUND(SUM(o.total_amount), 2) AS total_revenue
    FROM demo.ecommerce.orders o
    JOIN demo.ecommerce.customers c
        ON o.customer_id = c.customer_id
    GROUP BY DATE(o.order_timestamp), c.segment
    ORDER BY order_date, c.segment;

order_date,segment,total_orders,total_revenue
2026-07-15,Enterprise,140,129071.50
2026-07-15,Retail,66,58548.73
2026-07-15,SMB,58,65791.75
2026-07-15,Wholesale,54,53011.07
2026-08-20,Enterprise,130,130200.84
2026-08-20,Retail,76,70312.74
2026-08-20,SMB,67,61760.21
2026-08-20,Wholesale,51,49133.80
2026-09-10,Enterprise,141,146581.25
2026-09-10,Retail,79,77305.67


In [26]:
%%sql
Select * from demo.ecommerce.orders.partitions;

partition,spec_id,record_count,file_count,total_data_file_size_in_bytes,position_delete_record_count,position_delete_file_count,equality_delete_record_count,equality_delete_file_count,last_updated_at,last_updated_snapshot_id
"Row(order_timestamp_day=datetime.date(2026, 7, 15))",0,318,1,5242,0,0,0,0,2026-11-24 01:45:28.342000,575426542183088885
"Row(order_timestamp_day=datetime.date(2026, 8, 20))",0,324,1,5315,0,0,0,0,2026-11-24 01:45:28.342000,575426542183088885
"Row(order_timestamp_day=datetime.date(2026, 9, 10))",0,358,1,5619,0,0,0,0,2026-11-24 01:45:28.342000,575426542183088885


## INSPECT MEDATADA CHANGES

## INSPECT the metadata files using the Spark AVRO APIS

### Metadata 1 - Check for catalog pointer

==========================------------------------------------=========================================

## 2. Data Ingestion & Core DML

We will populate the table with an initial batch of raw events, perform row-level mutations (UPDATE and DELETE), and finally simulate an incremental batch arriving from a staging layer using a comprehensive `MERGE INTO` operation.

In [28]:
%%sql
--- 1. INGEST FEW REOCRDS

-- spark.sql("""
-- INSERT INTO demo.ecommerce.orders VALUES
--     (1001, 5501, CAST('2026-09-10 08:15:00' AS TIMESTAMP), 'PENDING', 120.50),
--     (1002, 5502, CAST('2026-09-10 09:30:00' AS TIMESTAMP), 'PENDING', 89.99),
--     (1003, 5503, CAST('2026-09-10 14:45:00' AS TIMESTAMP), 'PROCESSING', 1045.00)
-- """)

    
INSERT INTO demo.ecommerce.orders VALUES
    (1001, 5501, CAST('2026-09-10 08:15:00' AS TIMESTAMP), 'PENDING', 120.50),
    (1002, 5502, CAST('2026-09-10 09:30:00' AS TIMESTAMP), 'PENDING', 89.99),
    (1003, 5503, CAST('2026-09-10 14:45:00' AS TIMESTAMP), 'PROCESSING', 1045.00);



++
||
++
++

In [30]:
# 2. Row-level Update
spark.sql("""
UPDATE demo.ecommerce.orders 
SET status = 'PROCESSING' 
WHERE order_id = 1001
""")

DataFrame[]

#### NOW CHECK THE METADATA CHANGES

In [6]:


# 3. Row-level Delete
spark.sql("""
DELETE FROM demo.ecommerce.orders 
WHERE order_id = 1002
""")

# 4. MERGE INTO (Upsert) Operation
spark.sql("""
WITH staging_updates AS (
    SELECT 1001 AS order_id, 5501 AS customer_id, CAST('2026-09-10 08:15:00' AS TIMESTAMP) AS order_timestamp, 'COMPLETED' AS status, 120.50 AS total_amount UNION ALL
    SELECT 1004 AS order_id, 5504 AS customer_id, CAST('2026-09-13 10:00:00' AS TIMESTAMP) AS order_timestamp, 'PENDING' AS status, 45.00 AS total_amount
)
MERGE INTO demo.ecommerce.orders t
USING staging_updates s
ON t.order_id = s.order_id
WHEN MATCHED AND s.status = 'COMPLETED' THEN 
    UPDATE SET t.status = s.status
WHEN NOT MATCHED THEN 
    INSERT (order_id, customer_id, order_timestamp, status, total_amount)
    VALUES (s.order_id, s.customer_id, s.order_timestamp, s.status, s.total_amount)
""")

print("--- Final Table State after DML ---")
spark.sql("SELECT * FROM demo.ecommerce.orders ORDER BY order_id").show()

--- Final Table State after DML ---


+--------+-----------+-------------------+----------+------------+
|order_id|customer_id|    order_timestamp|    status|total_amount|
+--------+-----------+-------------------+----------+------------+
|       1|      63638|2026-09-11 01:29:37|PROCESSING|     4766.20|
|       2|      59939|2026-09-10 13:44:04| COMPLETED|      340.02|
|       3|      13485|2026-09-10 23:43:57| CANCELLED|      731.21|
|       4|       7685|2026-09-10 21:51:28| COMPLETED|     2137.73|
|       5|      85393|2026-09-11 18:12:12|   PENDING|       37.73|
|       6|      71678|2026-09-10 17:29:39|   PENDING|     2417.32|
|       7|      24740|2026-09-10 21:36:34| CANCELLED|     2010.05|
|       8|      13675|2026-09-10 20:09:31| CANCELLED|     4583.20|
|       9|      38696|2026-09-11 09:50:51|PROCESSING|      479.60|
|      10|      60516|2026-09-10 00:43:27| CANCELLED|     3164.42|
|      11|      55131|2026-09-11 02:53:54|PROCESSING|      419.19|
|      12|      14524|2026-09-10 12:31:38| CANCELLED|     1759

## 3. Metadata Inspection (Under the Hood)

Iceberg exposes its internal architectural state via system tables. Querying these tables allows you to track exactly how the metadata tree evaluates the active dataset.

*   `.history`: Chronological log of snapshots (commits).
*   `.snapshots`: Deep operational metrics (files added/removed).
*   `.files`: Physical Parquet objects distributed across object storage.
*   `.manifests`: Files that track data files and hold column-level min/max statistics for query pruning.

In [ ]:
print("--- 1. Table History ---")
spark.sql("SELECT made_current_at, snapshot_id, parent_id, is_current_ancestor FROM demo.ecommerce.orders.history").show(truncate=False)

print("--- 2. Snapshots ---")
spark.sql("SELECT snapshot_id, operation, summary FROM demo.ecommerce.orders.snapshots").show(truncate=False)

print("--- 3. Data & Delete Files ---")
spark.sql("SELECT file_path, file_format, record_count, file_size_in_bytes, partition FROM demo.ecommerce.orders.files").show(truncate=False)

print("--- 4. Manifests ---")
spark.sql("SELECT path, length, added_data_files_count, existing_data_files_count FROM demo.ecommerce.orders.manifests").show(truncate=False)

## 4. Time Travel & Versioning

Because Iceberg treats updates as immutable new snapshots, previous states of the table remain queryable. 
If a pipeline corrupts the main table, you can execute a hard rollback, which rewinds the table pointer to an explicit historical snapshot.

In [ ]:
# Dynamically fetch the first snapshot ID to demonstrate Time Travel
first_snapshot_id = spark.sql("SELECT snapshot_id FROM demo.ecommerce.orders.history ORDER BY made_current_at ASC LIMIT 1").collect()[0][0]

print(f"--- Time Travel: State at Snapshot {first_snapshot_id} ---")
spark.sql(f"SELECT * FROM demo.ecommerce.orders VERSION AS OF {first_snapshot_id}").show()

print("\n--- Rollback Command Syntax ---")
print("# To permanently undo changes, execute the system procedure:")
print(f"# spark.sql(\"CALL demo.system.rollback_to_snapshot('ecommerce.orders', {first_snapshot_id})\")")

## 5. Branching & Tagging (Data Lifecycle)

The **Write-Audit-Publish (WAP)** pattern enables logical isolation. Engineers can write massive ETL jobs to a branch without downstream consumers seeing the changes until they are verified and fast-forwarded to main.

Tags allow you to pin specific snapshots indefinitely for regulatory auditing.

In [ ]:
# 1. Create an isolated branch
spark.sql("ALTER TABLE demo.ecommerce.orders CREATE BRANCH IF NOT EXISTS etl_pipeline")

# 2. Write explicitly to the branch (Main remains unaffected)
spark.sql("""
INSERT INTO demo.ecommerce.orders.branch_etl_pipeline
VALUES (1005, 5505, CAST('2026-09-13 11:30:00' AS TIMESTAMP), 'PENDING', 210.00)
""")

print("--- ETL Branch Data (Includes order 1005) ---")
spark.sql("SELECT * FROM demo.ecommerce.orders VERSION AS OF 'etl_pipeline'").show()

# 3. Fast-forward main branch to publish ETL changes
spark.sql("CALL demo.system.fast_forward('ecommerce.orders', 'main', 'etl_pipeline')")
print("--- Main Branch After Fast-Forward ---")
spark.sql("SELECT * FROM demo.ecommerce.orders ORDER BY order_id").show()

# 4. Create an audit tag
spark.sql("ALTER TABLE demo.ecommerce.orders CREATE TAG IF NOT EXISTS end_of_q3_2026 RETAIN 365 DAYS")
print("Audit Tag 'end_of_q3_2026' created successfully.")

## 6. Table Maintenance & Optimization

Streaming ingestion and MoR fragment the table into thousands of small Parquet files. Routine maintenance is mandatory.
*   **rewrite_data_files**: Consolidates small files into optimized target sizes.
*   **rewrite_manifests**: Aligns the metadata tree with the new data file structure.
*   **expire_snapshots**: Removes old snapshot references to manage storage costs.
*   **remove_orphan_files**: Safely sweeps physical files no longer tracked by metadata.

In [ ]:
print("1. Running Binpack Compaction...")
spark.sql("""
CALL demo.system.rewrite_data_files(
  table => 'ecommerce.orders', 
  strategy => 'binpack', 
  options => map('target-file-size-bytes', '536870912')
)
""").show()

print("2. Rewriting Manifests...")
spark.sql("CALL demo.system.rewrite_manifests('ecommerce.orders')").show()

print("3. Expiring historical snapshots (Retaining the last 2)...")
spark.sql("""
CALL demo.system.expire_snapshots(
  table => 'ecommerce.orders', 
  retain_last => 2
)
""").show()

print("4. Sweeping Orphan Files...")
spark.sql("""
CALL demo.system.remove_orphan_files(
  table => 'ecommerce.orders'
)
""").show()

print("Maintenance Complete! Your table is fully optimized.")